In [1]:
import sys
!{sys.executable} -m pip install pandas


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import ast
import pandas as pd
import json
import numpy as np
from neo4j import GraphDatabase
import regex

In [3]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

In [4]:
uri = "bolt://neo4j-gds-apoc-n10s:7687"
username = "neo4j"
password = "neo4jpassword"

In [5]:
driver = GraphDatabase.driver(uri, auth=(username, password))

In [6]:
DATA_DIR = "notebooks/rdb"

In [7]:
def type_cast(input_data):
    new_data = None
    if pd.isna(input_data):
        input_data = "[]"
    elif '[' in input_data and ']' in input_data and 'nan' in input_data:
        input_data = input_data.replace("nan", "")
    
    new_data = ast.literal_eval(input_data)

    return new_data

### neo4j 초기화
- 아래 코드를 통해 넣을 수 없는 데이터(다른 방법으로 이미 넣어둔 데이터)가 있는 경우 아래 코드는 실행하면 안 됨

In [8]:
with driver.session() as session:
    # 모든 관계와 노드 제거
    session.run("MATCH (n) DETACH DELETE n")

### 데이터 로딩
- 스키마 참고
  - https://confluence.tde.sktelecom.com/pages/viewpage.action?pageId=734203873
- 순서
  - PRODUCT + PRICE, VOICE, SMS, DATA, TOPUP, CUSTOMERCONDITION, BENEFITCONDITION, DEDUCTIBLE
  - product_group
  - relation_db

### PRODUCT + PRICE

In [9]:
product_table = pd.read_csv(os.path.join(DATA_DIR, "PRODUCT.csv"))
price_table = pd.read_csv(os.path.join(DATA_DIR, "PRICE.csv"))
product_table = product_table.merge(price_table, on="pmProductID", how="left")

In [10]:
# 리스트형으로 변환
product_table["generation"] = product_table["generation"].apply(lambda x: type_cast(x))
product_table["marketingKeyword"] = product_table["marketingKeyword"].apply(lambda x: type_cast(x))
product_table["mappedProductCode"] = product_table["mappedProductCode"].apply(lambda x: type_cast(x))

# None -> "null"
product_table["productSubscriptionCondition"] = product_table["productSubscriptionCondition"].fillna("null")

In [11]:
eng_colnames = [
    'pmProductID', 'mappedProductCode', 'generation', 'marketingKeyword', 
    'productName', 'productNameInEnglish', 'lineup', 'classifiedGroup',
    'productDescription', 'productSubscriptionCondition', 'statusOfOperation',
    'monthlyPrice', 'monthlyPriceWithoutVAT', 'monthlyPriceWithSelectableInstallment', 
    'billingMethod', 'netPrice'
]

kor_colnames = [
    '고유ID', '상품코드매핑', '통신규격', '마케팅키워드',
    '상품명', '영문상품명', '라인업', '상품분류',
    '상품설명', '상품가입조건', '운영상태',
    '월정액', '부가세제외월정액', '선택약정할인포함부가세제외월정액', 
    '청구방법', 'net가격'
]

kor_cols_map = {x:y for x, y in zip(eng_colnames, kor_colnames)}

product_table = product_table.rename(columns=kor_cols_map)

### VOICE

In [12]:
voice_table = pd.read_csv(os.path.join(DATA_DIR, "VOICE.csv"))

# 리스트형으로 변환
voice_table["refillRange"] = voice_table["refillRange"].apply(lambda x: type_cast(x))

# refillAmount 정규화
voice_table["refillAmountRatio"] = voice_table["refillAmount"].apply(lambda x: int(x.replace("%", ""))*0.01 if pd.notna(x) else 0)

# None -> "null"
voice_table["includedVoiceCallTospecifiedNumbers"] = voice_table["includedVoiceCallTospecifiedNumbers"].fillna("null")

In [13]:
del voice_table["refillAmount"]

In [14]:
eng_colnames = [
    'pmProductID',
    'includedVoiceCall',
    'includedVideoOrValueAddedCall',
    'includedVoiceCallTospecifiedNumbers',
    'refillAmountRatio',
    'refillRange'
]

kor_colnames = [
    '고유ID',
    '음성통화제공량',
    '영상및부가통화제공량',
    '지정번호통화제공량',
    '리필비율한도',
    '리필대상'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [15]:
voice_table = voice_table.rename(columns=kor_cols_map)

### SMS


In [16]:
sms_table = pd.read_csv(os.path.join(DATA_DIR, "SMS.csv"))

In [17]:
# 컬럼명 매핑 정의
eng_colnames = [
    'pmProductID',
    'includedText',
    'textRange'
]

kor_colnames = [
    '고유ID',
    '문자제공량',
    '문자대상'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

sms_table = sms_table.rename(columns=kor_cols_map)

### DATA


In [18]:
data_table = pd.read_csv(os.path.join(DATA_DIR, "DATA.csv"))

In [19]:
data_table["maximumShareAmount"] = data_table["maximumShareAmount"].apply(lambda x: float(x.replace("GB", "")) if pd.notna(x) else None)

# data_table에서 NaN을 칼럼의 데이터타입에 맞춰 임의의 값으로 치환
data_table["includedData"] = data_table["includedData"].fillna(0.0)
data_table["includedDataForSharingAndTethering"] = data_table["includedDataForSharingAndTethering"].fillna(0.0)
data_table["includedMVoIP"] = data_table["includedMVoIP"].fillna(0.0)
data_table["appliedSpeed"] = data_table["appliedSpeed"].fillna(0.0)
data_table["seniorDataExceedAvailable"] = data_table["seniorDataExceedAvailable"].fillna("null")
data_table["generalDataExceedAvailable"] = data_table["generalDataExceedAvailable"].fillna("null")
data_table["dataRefillAmount"] = data_table["dataRefillAmount"].fillna(0.0)
data_table["dataRefillCouponGiftingAvailability"] = data_table["dataRefillCouponGiftingAvailability"].fillna("null")
data_table["maximumShareAmount"] = data_table["maximumShareAmount"].fillna(0.0)
data_table["dataGiftReceivingAvailability"] = data_table["dataGiftReceivingAvailability"].fillna("null")

In [20]:
# 컬럼명 매핑 정의
eng_colnames = [
    'pmProductID',
    'includedData', 'includedDataForSharingAndTethering',
    'includedMVoIP', 'appliedSpeed', 'seniorDataExceedAvailable',
    'generalDataExceedAvailable', 'dataRefillAmount',
    'dataRefillCouponGiftingAvailability', 'maximumShareAmount',
    'dataGiftReceivingAvailability'
]

kor_colnames = [
    '고유ID',
    '기본제공데이터용량', '기본제공데이터중공유가능용량',
    '기본제공데이터중mvoip용량', '데이터소진후데이터제공속도', '시니어대상데이터소진후최대금액및속도제한적용',
    '데이터소진후최대금액및속도제한적용', '데이터리필가능용량',
    '데이터리필쿠폰선물가능여부', '최대데이터선물가능용량',
    '데이터선물받기가능여부'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

data_table = data_table.rename(columns=kor_cols_map)

### TOPUP


In [21]:
topup_table = pd.read_csv(os.path.join(DATA_DIR, "TOPUP.csv"))

In [22]:
# NaN 처리
topup_table["minimumChargeAmount"] = topup_table["minimumChargeAmount"].fillna(0.0)
topup_table["maximumChargeAmount"] = topup_table["maximumChargeAmount"].fillna(0.0)

In [23]:
# 컬럼명 매핑 정의
eng_colnames = [
    'pmProductID', 'reChargeAvailability', 'minimumChargeAmount', 'maximumChargeAmount'
]

kor_colnames = [
    '고유ID', '충전서비스대상여부', '최소충전금액', '최대충전금액'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

topup_table = topup_table.rename(columns=kor_cols_map)

### CUSTOMERCONDITION


In [24]:
condition_table = pd.read_csv(os.path.join(DATA_DIR, "CUSTOMERCONDITION.csv"))

In [25]:
# ageRule 파싱
ageRule_table = []
for _, row in condition_table.iterrows():
    ageRule = ast.literal_eval(row['ageRule'])
    if not ageRule:
        ageRule_table.append(["null", 0, "null", 999])
    else:
        # 초기값 (기본값)
        minAgeCriteria = "null"
        minAge = 0
        maxAgeCriteria = "null"
        maxAge = 999
        for rule in ageRule:
            if "이하" in rule["value"]:
                maxAge = int(regex.findall(r"\d+", rule["value"])[0])
                maxAgeCriteria = regex.findall(r"[일월]기준", rule["value"])[0]
            elif "이상" in rule["value"]:
                minAge = int(regex.findall(r"\d+", rule["value"])[0])
                minAgeCriteria = regex.findall(r"[일월]기준", rule["value"])[0]
        ageRule_table.append([minAgeCriteria, minAge, maxAgeCriteria, maxAge])

condition_table = pd.concat([condition_table, pd.DataFrame(ageRule_table, columns=["minAgeCriteria", "minAge", "maxAgeCriteria", "maxAge"])], axis=1)

In [26]:
condition_table.drop(columns=['ageRule'], inplace=True)

In [27]:
# 리스트형으로 변환
condition_table["customerTypeValueList"] = condition_table["customerTypeValueList"].apply(lambda x: type_cast(x))
condition_table["individualCustomerSubtypeValueList"] = condition_table["individualCustomerSubtypeValueList"].apply(lambda x: type_cast(x))
condition_table["duplicateNameOnboardGroupList"] = condition_table["duplicateNameOnboardGroupList"].apply(lambda x: type_cast(x))

# NaN 처리
condition_table["customerTypeEligibility"] = condition_table["customerTypeEligibility"].fillna("null")
condition_table["individualCustomerSubtypeEligibility"] = condition_table["individualCustomerSubtypeEligibility"].fillna("null")
condition_table["directPlanOnboard"] = condition_table["directPlanOnboard"].fillna("null")
condition_table["fixedPlanContractConcurrentSignupRestriction"] = condition_table["fixedPlanContractConcurrentSignupRestriction"].fillna("null")
condition_table["tsupportFundOnboard"] = condition_table["tsupportFundOnboard"].fillna("null")
condition_table["duplicateNameOnboardEligibility"] = condition_table["duplicateNameOnboardEligibility"].fillna("null")
condition_table["specialCustomerIsSoldier"] = condition_table["specialCustomerIsSoldier"].fillna("null")
condition_table["specialCustomerIsSoldier"] = condition_table["specialCustomerIsSoldier"].fillna("null")

In [28]:
eng_colnames = [
    'pmProductID',
    'customerTypeEligibility', 'customerTypeValueList',
    'individualCustomerSubtypeEligibility',
    'individualCustomerSubtypeValueList', 'directPlanOnboard',
    'fixedPlanContractConcurrentSignupRestriction', 'tsupportFundOnboard',
    'duplicateNameOnboardEligibility', 'duplicateNameOnboardGroupList',
    'specialCustomerIsSoldier', 'minAgeCriteria', 'minAge',
    'maxAgeCriteria', 'maxAge'
]

kor_colnames = [
    '고유ID',
    '고객유형별가입가능여부', '고객유형목록',
    '개인고객세부유형별가입가능여부', 
    '개인고객세부유형목록', '다이렉트플랜가입가능여부',
    '선택약정동시가입가능여부', 'T지원금약정동시가입가능여부',
    '동일명의가입가능여부', '동일명의가입불가그룹목록',
    '군인전용요금제여부', '가입가능최소나이계산기준', '가입가능최소나이',
    '최대나이계산기준', '가입가능최대나이'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [29]:
condition_table = condition_table.rename(columns=kor_cols_map)

### BENEFITCONDITION
- 혜택 조건 정보 -> 구체적인 혜택 정보가 없어서 넣는 의미가 없어보여서 일단 스킵

### DEDUCTIBLE

In [30]:
deductible_table = pd.read_csv(os.path.join(DATA_DIR, "DEDUCTIBLE.csv"))

In [31]:
deductible_table

,pmProductID,deductibilityForDisability,additionalOfferForDisabilities
0,PA00000001,N,NaN
1,PA00000002,Y,200분
2,PA00000003,N,NaN
3,PA00000004,N,NaN
4,PA00000005,N,NaN
...,...,...,...
120,PA00002811,N,NaN
121,PA00002812,N,NaN
122,PA00002813,N,NaN
123,PA00002814,N,NaN


In [32]:
# Normalize
deductible_table["additionalOfferForDisabilities"] = deductible_table["additionalOfferForDisabilities"].apply(lambda x: int(x.replace("분", "")) if pd.notna(x) else 0)

In [33]:
eng_colnames = ['pmProductID', 'deductibilityForDisability', 'additionalOfferForDisabilities']
kor_colnames = ['고유ID', '장애인혜택제공여부', '장애인부가통화추가제공량']

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

deductible_table = deductible_table.rename(columns=kor_cols_map)

### Join tables
- 위에서 생성한 테이블들을 하나로 join
- Join 대상 테이블들
  - PRODUCT + PRICE, VOICE, SMS, DATA, TOPUP, CUSTOMERCONDITION, BENEFITCONDITION, DEDUCTIBLE

In [34]:
merged_table = (
    product_table.merge(voice_table, on="고유ID", how="left")
    .merge(sms_table, on="고유ID", how="left")
    .merge(data_table, on="고유ID", how="left")
    .merge(topup_table, on="고유ID", how="left")
    .merge(condition_table, on="고유ID", how="left")
    .merge(deductible_table, on="고유ID", how="left")
)

### 그래프DB에 저장

In [35]:
create_query = """
MERGE (p:`요금제` {고유ID: $고유ID})
SET p += $props
"""

with driver.session() as session:
    for _, row in merged_table.iterrows():
        props = row.to_dict()
        for k, v in props.items():
            if isinstance(v, float) and np.isnan(v):
                props[k] = "null"
                print(k, "!")
        session.run(create_query, 고유ID=row["고유ID"], props=props)


### 관계 생성
- relation_db
- product_group

In [38]:
relation_db_table = pd.read_csv(os.path.join(DATA_DIR, "relation_db.csv"))

relation_db_table = relation_db_table[(relation_db_table["productId"].notna())&(relation_db_table["productId"] != "-")].copy()

relation_type_translation = {
    'productBenefitConditions.allBenefitList': "할인", #부가서비스할인
    'optionData.dataOptionProvidingMethod': "데이터충전", #데이터충전혜택
    'productRelation.signupConcurrentTermination.productList': "가입동시해지",
    'productRelation.signupPreTermination.productList': "가입이전해지",
    'productRelation.terminationConcurrentTermination.productList': "해지동시해지",
    'productRelation.terminationPreTermination.productList': "해지이전해지"
}

eng_colnames = ["productId", "productName", "type"]
kor_colnames = ["부가서비스ID", "부가서비스명", "관계유형"]
kor_cols_map  = {x: y for x, y in zip(eng_colnames, kor_colnames)}

# 그래프에 저장
with driver.session() as session:
    for _, row in relation_db_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}
        relation_type = relation_type_translation[row['type']]

        if relation_type:
            session.run(
                f"""
                MATCH (p:요금제 {{고유ID: $product_id}})
                MERGE (b:부가서비스 {{부가서비스ID: $부가서비스ID, 부가서비스명: $부가서비스명}})
                MERGE (p)-[:{relation_type}]->(b)
                """,
                product_id=row['pmProductID'],
                부가서비스ID=properties['부가서비스ID'],
                부가서비스명=properties['부가서비스명']
            )


In [ ]:
product_group_table = pd.read_csv(os.path.join(DATA_DIR, "product_group.csv"))

In [40]:
product_group_table

,groupName,pmProductId,legacyProductId,productName
0,TING_PRCPLN,PA00000015,NA00006157,0플랜 라지
1,TING_PRCPLN,PA00000018,NA00008272,0 청년 59 100GB업
2,TING_PRCPLN,PA00000019,NA00008274,0 청년 59 36GB업
3,TING_PRCPLN,PA00000020,NA00008273,0 청년 59 60GB업
4,TING_PRCPLN,PA00000021,NA00008275,0 청년 59 15GB업
...,...,...,...,...
141,스마트기기 요금제,PA00002813,NA00009101,다이렉트5G 76(스마트기기)
142,유튜프 프리미엄 요금제,PA00002804,NA00009122,다이렉트5G 76(유튜브 프리미엄)
143,유튜프 프리미엄 요금제,PA00002808,NA00009121,5GX 프리미엄(유튜브 프리미엄)
144,유튜프 프리미엄 요금제,PA00000694,NA00009120,5GX 플래티넘(유튜브 프리미엄)


In [41]:
eng_colnames = ['groupName']
kor_colnames = ['그룹명']

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [42]:
# 그래프에 저장
with driver.session() as session:
    for _, row in product_group_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}

        # 그룹 노드 생성 및 요금제와 연결
        session.run(
            """
            MERGE (g:요금제그룹 {그룹명: $그룹명})
            WITH g
            MATCH (p:요금제 {고유ID: $pmProductId})
            MERGE (p)-[:속함]->(g)
            """,
            그룹명=properties['그룹명'],
            pmProductId=row['pmProductId']
        )